### Rain-on-snow project <br>
#### Definition of ROS zone and events accross NE US

In [2]:
import sys
# Path to ROS functions
sys.path.append('../') # Place path to where your ros-workflow directory is

In [3]:
import os
import functions.rain_on_snow_fncns as ros
from functions.rain_on_snow_fncns import nwm_proj
import geopandas
import pandas as pd

In [3]:
from dask.distributed import Client
client = Client()
#clien = Client(n_workers=6, memory_limit='2GB') 
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 16,Total memory: 251.20 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:37255,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:34709,Total threads: 4
Dashboard: http://127.0.0.1:45193/status,Memory: 62.80 GiB
Nanny: tcp://127.0.0.1:38941,


1. Load NWM data

In [4]:
# Path to AWS NWM retrospective data
s3_path = 's3://noaa-nwm-retrospective-3-0-pds/CONUS/zarr/ldasout.zarr'
#timerange_test = slice('2019-01-01 00:00:00','2020-12-31 18:00:00')
timerange_test = slice('1979-10-01 00:00:00','2022-09-30 18:00:00')
variables = ["SNEQV", "QRAIN"]

In [ ]:
%%time
# NE domain (NWM LCC meters): x 1.4e6 -> east edge, y 0 -> 1.5e6
ds_ne = ros.read_nwmData(awsPath = s3_path, variables = variables, timerange = timerange_test, x_range = (1.4e6, None), y_range = (0, 1.5e6))

2. Load study locations

In [6]:
# Load GAGES II basins
shpPath = '~/netfiles/ciroh/mmorales/ROS_project/shapefiles/GAGES_II/boundaries_shapefiles_by_aggeco/boundaries-shapefiles-by-aggeco/'
fname_ref = 'bas_ref_all.shp'
fname_nonref = 'bas_nonref_NorthEast.shp'

shapefile_ref = f'{shpPath}{fname_ref}'
shapefile_nonref = f'{shpPath}{fname_nonref}'

In [7]:
# Prepares basins shapefile in the NWM projection and subsets them to the domain extension
ref_bsns_shp = ros.prepare_spatial_assets(ds = ds_ne, shp_path = shapefile_ref)
nonref_bsns_shp = ros.prepare_spatial_assets(ds = ds_ne, shp_path = shapefile_nonref)

Automatic Setup Complete: 304 basins selected.
Automatic Setup Complete: 609 basins selected.


In [8]:
shp_dict = {
    "gagesii_bsn_ref": ref_bsns_shp,
    "gagesii_bsn_nonref": nonref_bsns_shp
}

3. Extract daily ROS events

In [9]:
# Load pre-computed ROS events from notebook 1 outputs
saveDir = './output'

events_dict = {
    name: pd.read_parquet(f'{saveDir}/ros_events_{name}.parquet')
    for name in shp_dict
}

for name, df in events_dict.items():
    print(f"{name}: {len(df):,} event records, {df['GAGE_ID'].nunique()} basins")

gagesii_bsn_ref: 52,326 event records, 304 basins
gagesii_bsn_nonref: 150,313 event records, 609 basins


4. Extracting daily hydrological properties

In [ ]:
%%time
# Extract NWM variable means over each basin for all ROS event timesteps (NWM is 3-hourly).
# Basin pixel-coverage weights are precomputed once, so each timestep's basin means are a
# fast sparse matmul rather than a repeated exactextract pass. Partial-pixel (fractional)
# coverage is preserved -- results are identical to exactextract's 'mean' op.
# Block size is auto-tuned to ~3 GB reads (float32); pass read_dtype='float64' for
# bit-closer agreement at 2x memory.
variables = ['QRAIN', 'SNEQV']

for name, shp in shp_dict.items():
    print(f"\n{'='*55}")
    print(f"Processing {name}...")
    out_path = f'{saveDir}/hrly_ros_hydroprop_{name}.parquet'

    ros.extract_hydrologic_prop(
        ds=ds_ne,
        variables=variables,
        events_df=events_dict[name],
        shp=shp,
        output_path=out_path
    )
    print(f"  Saved → {out_path}")

In [ ]:
client.shutdown()